In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\AmalDev\Downloads\obd-driver-behavior\exp1_14drivers_14cars_dailyRoutes.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nNulls:\n", df.isnull().sum())
print("\nSample:\n", df.head(5))
print("\nStats:\n", df.describe().round(2))

C:\Users\AmalDev\AppData\Local\Temp\ipykernel_26552\4286344646.py:3: DtypeWarning: Columns (1,2,4,5,6,9,10,14,15,16,20,21,22,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\AmalDev\Downloads\obd-driver-behavior\exp1_14drivers_14cars_dailyRoutes.csv')


Shape: (60439, 33)

Columns: ['TIMESTAMP', 'MARK', 'MODEL', 'CAR_YEAR', 'ENGINE_POWER', 'AUTOMATIC', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE(KPA)', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE', 'MAF', 'LONG TERM FUEL TRIM BANK 2', 'FUEL_TYPE', 'AIR_INTAKE_TEMP', 'FUEL_PRESSURE', 'SPEED', 'SHORT TERM FUEL TRIM BANK 2', 'SHORT TERM FUEL TRIM BANK 1', 'ENGINE_RUNTIME', 'THROTTLE_POS', 'DTC_NUMBER', 'TROUBLE_CODES', 'TIMING_ADVANCE', 'EQUIV_RATIO', 'MIN', 'HOURS', 'DAYS_OF_WEEK', 'MONTHS', 'YEAR']

Dtypes:
 TIMESTAMP                      float64
MARK                            object
MODEL                           object
CAR_YEAR                       float64
ENGINE_POWER                    object
AUTOMATIC                       object
VEHICLE_ID                      object
BAROMETRIC_PRESSURE(KPA)       float64
ENGINE_COOLANT_TEMP            float64
FUEL_LEVEL                      object
ENGINE_LOAD                     object

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r'C:\Users\AmalDev\Downloads\obd-driver-behavior\exp1_14drivers_14cars_dailyRoutes.csv',
    low_memory=False
)

# ── STEP 1: Convert TIMESTAMP from Unix ms to datetime ──────────────────
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], unit='ms', errors='coerce')
print("Timestamp sample:", df['TIMESTAMP'].head(3).values)

# ── STEP 2: Strip % signs and European commas → convert to float ─────────
def clean_percent(col):
    return pd.to_numeric(
        col.astype(str).str.replace('%','').str.replace(',','.').str.strip(),
        errors='coerce'
    )

df['THROTTLE_POS']    = clean_percent(df['THROTTLE_POS'])
df['TIMING_ADVANCE']  = clean_percent(df['TIMING_ADVANCE'])
df['EQUIV_RATIO']     = clean_percent(df['EQUIV_RATIO'])
df['FUEL_LEVEL']      = clean_percent(df['FUEL_LEVEL'])
df['ENGINE_LOAD']     = clean_percent(df['ENGINE_LOAD'])
df['ENGINE_POWER']    = clean_percent(df['ENGINE_POWER'])
df['MAF']             = pd.to_numeric(df['MAF'].astype(str).str.replace(',','.'), errors='coerce')
df['SHORT TERM FUEL TRIM BANK 1'] = clean_percent(df['SHORT TERM FUEL TRIM BANK 1'])
df['SHORT TERM FUEL TRIM BANK 2'] = clean_percent(df['SHORT TERM FUEL TRIM BANK 2'])
df['LONG TERM FUEL TRIM BANK 2']  = clean_percent(df['LONG TERM FUEL TRIM BANK 2'])
df['ENGINE_RUNTIME']  = pd.to_numeric(df['ENGINE_RUNTIME'], errors='coerce')

print("\nThrottle sample:", df['THROTTLE_POS'].dropna().head(5).values)
print("Timing sample:", df['TIMING_ADVANCE'].dropna().head(5).values)

# ── STEP 3: Parse DTC_NUMBER → extract fault count as integer ────────────
df['FAULT_COUNT'] = df['DTC_NUMBER'].astype(str)\
    .str.extract(r'(\d+)\s*codes?', expand=False)\
    .astype(float)
df['HAS_FAULT'] = (df['FAULT_COUNT'] > 0).astype(int)
print("\nFault count distribution:\n", df['FAULT_COUNT'].value_counts())

# ── STEP 4: Drop useless columns ─────────────────────────────────────────
cols_to_drop = [
    'FUEL_PRESSURE',        # constant 48.0, zero variance
    'TROUBLE_CODES',        # 80% null, redundant with DTC_NUMBER
    'DTC_NUMBER',           # replaced by FAULT_COUNT
    'FUEL_TYPE',            # 67% null, not useful for behavior analysis
]
df = df.drop(columns=cols_to_drop)
print("\nRemaining columns:", len(df.columns))

# ── STEP 5: Flag idle rows (engine on but not moving) ────────────────────
df['IS_IDLE']   = ((df['SPEED'] == 0) & (df['ENGINE_RPM'] > 0)).astype(int)
df['IS_MOVING'] = (df['SPEED'] > 5).astype(int)

# ── STEP 6: Drop columns with >80% nulls globally ────────────────────────
null_pct = df.isnull().mean()
cols_high_null = null_pct[null_pct > 0.80].index.tolist()
print("\nDropping >80% null columns:", cols_high_null)
df = df.drop(columns=cols_high_null)

# ── STEP 7: Reconstruct full TIMESTAMP from parts where missing ──────────
# Fill missing TIMESTAMP using YEAR, MONTHS, DAYS_OF_WEEK, HOURS, MIN
print("\nFinal shape:", df.shape)
print("Remaining nulls:\n", df.isnull().sum().sort_values(ascending=False).head(10))

Timestamp sample: ['2017-08-16T16:55:04.267000000' '2017-08-16T16:55:12.283000000'
 '2017-08-16T16:55:20.291000000']

Throttle sample: [25. 25. 25. 25. 25.]
Timing sample: [56.9 56.5 57.3 56.5 56.9]

Fault count distribution:
 FAULT_COUNT
0.0      41065
1.0       6071
107.0        2
16.0         2
65.0         1
8.0          1
Name: count, dtype: int64

Remaining columns: 31

Dropping >80% null columns: ['BAROMETRIC_PRESSURE(KPA)', 'FUEL_LEVEL', 'AMBIENT_AIR_TEMP', 'MAF', 'ENGINE_RUNTIME', 'EQUIV_RATIO']

Final shape: (60439, 27)
Remaining nulls:
 LONG TERM FUEL TRIM BANK 2     47369
SHORT TERM FUEL TRIM BANK 2    47369
INTAKE_MANIFOLD_PRESSURE       35350
ENGINE_LOAD                    29467
THROTTLE_POS                   26580
ENGINE_RPM                     26580
ENGINE_COOLANT_TEMP            26475
TIMING_ADVANCE                 26277
AIR_INTAKE_TEMP                26087
SHORT TERM FUEL TRIM BANK 1    22844
dtype: int64


In [3]:
# ── STEP 8: Cap impossible fault counts ──────────────────────────────────
df['FAULT_COUNT'] = df['FAULT_COUNT'].clip(upper=10)
print("Fault count after cap:\n", df['FAULT_COUNT'].value_counts())

# ── STEP 9: Handle remaining nulls smartly ───────────────────────────────
# LONG/SHORT TERM FUEL TRIM — 78% null → drop both
df = df.drop(columns=['LONG TERM FUEL TRIM BANK 2', 'SHORT TERM FUEL TRIM BANK 2'])

# Remaining sensor nulls — fill with median per VEHICLE_ID
sensor_cols = ['ENGINE_RPM', 'ENGINE_LOAD', 'THROTTLE_POS',
               'ENGINE_COOLANT_TEMP', 'TIMING_ADVANCE',
               'AIR_INTAKE_TEMP', 'INTAKE_MANIFOLD_PRESSURE',
               'SHORT TERM FUEL TRIM BANK 1']

for col in sensor_cols:
    df[col] = df.groupby('VEHICLE_ID')[col]\
                .transform(lambda x: x.fillna(x.median()))

# ── STEP 10: Fix SPEED outliers ──────────────────────────────────────────
print("\nSpeed outliers (>120 km/h):", (df['SPEED'] > 120).sum())
df['SPEED'] = df['SPEED'].clip(upper=120)

# ── STEP 11: Create driving behaviour label ───────────────────────────────
def label_behavior(row):
    if row['SPEED'] <= 5:
        return 'IDLE'
    elif row['SPEED'] <= 40 and row['ENGINE_RPM'] <= 1500:
        return 'ECO'
    elif row['SPEED'] <= 80 and row['ENGINE_RPM'] <= 2500:
        return 'NORMAL'
    else:
        return 'AGGRESSIVE'

df['DRIVING_STYLE'] = df.apply(label_behavior, axis=1)
print("\nDriving style distribution:\n", df['DRIVING_STYLE'].value_counts())

# ── STEP 12: Engineer key features ───────────────────────────────────────
df['RPM_PER_SPEED']    = (df['ENGINE_RPM'] / (df['SPEED'] + 1)).round(2)
df['THROTTLE_LOAD_RATIO'] = (df['THROTTLE_POS'] / (df['ENGINE_LOAD'] + 1)).round(2)
df['TRIP_HOUR']        = df['TIMESTAMP'].dt.hour
df['IS_PEAK_HOUR']     = df['TRIP_HOUR'].apply(lambda x: 1 if x in range(8,10) or x in range(17,20) else 0)

# ── STEP 13: Save cleaned file ────────────────────────────────────────────
df.to_csv(r'C:\Users\AmalDev\Downloads\obd-driver-behavior\obd_cleaned.csv', index=False)

print("\n✅ CLEANING COMPLETE")
print("Final shape:", df.shape)
print("Final columns:", df.columns.tolist())
print("\nDriving style counts:\n", df['DRIVING_STYLE'].value_counts())
print("\nNull check:\n", df.isnull().sum().sum(), "total nulls remaining")

Fault count after cap:
 FAULT_COUNT
0.0     41065
1.0      6071
10.0        5
8.0         1
Name: count, dtype: int64

Speed outliers (>120 km/h): 192

Driving style distribution:
 DRIVING_STYLE
IDLE          20535
AGGRESSIVE    17338
NORMAL        14505
ECO            8061
Name: count, dtype: int64

✅ CLEANING COMPLETE
Final shape: (60439, 30)
Final columns: ['TIMESTAMP', 'MARK', 'MODEL', 'CAR_YEAR', 'ENGINE_POWER', 'AUTOMATIC', 'VEHICLE_ID', 'ENGINE_COOLANT_TEMP', 'ENGINE_LOAD', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE', 'AIR_INTAKE_TEMP', 'SPEED', 'SHORT TERM FUEL TRIM BANK 1', 'THROTTLE_POS', 'TIMING_ADVANCE', 'MIN', 'HOURS', 'DAYS_OF_WEEK', 'MONTHS', 'YEAR', 'FAULT_COUNT', 'HAS_FAULT', 'IS_IDLE', 'IS_MOVING', 'DRIVING_STYLE', 'RPM_PER_SPEED', 'THROTTLE_LOAD_RATIO', 'TRIP_HOUR', 'IS_PEAK_HOUR']

Driving style counts:
 DRIVING_STYLE
IDLE          20535
AGGRESSIVE    17338
NORMAL        14505
ECO            8061
Name: count, dtype: int64

Null check:
 341287 total nulls remaining


In [8]:
# Fix remaining nulls — use global median as fallback
sensor_cols = ['ENGINE_RPM', 'ENGINE_LOAD', 'THROTTLE_POS',
               'ENGINE_COOLANT_TEMP', 'TIMING_ADVANCE',
               'AIR_INTAKE_TEMP', 'INTAKE_MANIFOLD_PRESSURE',
               'SHORT TERM FUEL TRIM BANK 1', 'RPM_PER_SPEED',
               'THROTTLE_LOAD_RATIO']

for col in sensor_cols:
    global_median = df[col].median()
    df[col] = df[col].fillna(global_median)

# Fill remaining nulls in non-sensor columns
df['SPEED']       = df['SPEED'].fillna(0)
df['FAULT_COUNT'] = df['FAULT_COUNT'].fillna(0)
df['HAS_FAULT']   = df['HAS_FAULT'].fillna(0)

# Check
remaining = df.isnull().sum()
print("Nulls per column:")
print(remaining[remaining > 0])
print("\nTotal nulls:", df.isnull().sum().sum())
print("Shape:", df.shape)

# Save final cleaned file
df.to_csv(r'C:\Users\AmalDev\Downloads\obd-driver-behavior\obd_cleaned.csv', index=False)
print("\n✅ Saved: obd_cleaned.csv")

Nulls per column:
TIMESTAMP       12925
MARK            12980
MODEL           12980
CAR_YEAR        12980
ENGINE_POWER    12980
AUTOMATIC       12980
VEHICLE_ID      12925
MIN             12927
HOURS           12927
DAYS_OF_WEEK    12927
MONTHS          12927
YEAR            12927
TRIP_HOUR       12925
dtype: int64

Total nulls: 168310
Shape: (60439, 30)

✅ Saved: obd_cleaned.csv


In [10]:
# Drop rows where VEHICLE_ID is null — no identity = unusable row
df = df.dropna(subset=['VEHICLE_ID', 'TIMESTAMP'])

# Fill MARK/MODEL/CAR_YEAR/ENGINE_POWER/AUTOMATIC 
# using forward fill within each vehicle group
identity_cols = ['MARK', 'MODEL', 'CAR_YEAR', 'ENGINE_POWER', 'AUTOMATIC']
df = df.sort_values(['VEHICLE_ID', 'TIMESTAMP'])
df[identity_cols] = df.groupby('VEHICLE_ID')[identity_cols].ffill()

# Rebuild time columns from TIMESTAMP (since MIN/HOURS/MONTHS/YEAR had nulls)
df['MIN']          = df['TIMESTAMP'].dt.minute
df['HOURS']        = df['TIMESTAMP'].dt.hour
df['DAYS_OF_WEEK'] = df['TIMESTAMP'].dt.dayofweek
df['MONTHS']       = df['TIMESTAMP'].dt.month
df['YEAR']         = df['TIMESTAMP'].dt.year
df['TRIP_HOUR']    = df['TIMESTAMP'].dt.hour

# Final check
print("Total nulls:", df.isnull().sum().sum())
print("Shape:", df.shape)
print("\nNulls per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Save final
df.to_csv(
    r'C:\Users\AmalDev\Downloads\obd-driver-behavior\obd_cleaned.csv',
    index=False
)
print("\n✅ FINAL CLEAN FILE SAVED")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nDriving style distribution:")
print(df['DRIVING_STYLE'].value_counts())
print("\nVehicles:", df['VEHICLE_ID'].nunique())

Total nulls: 275
Shape: (47514, 30)

Nulls per column:
MARK            55
MODEL           55
CAR_YEAR        55
ENGINE_POWER    55
AUTOMATIC       55
dtype: int64

✅ FINAL CLEAN FILE SAVED
Rows: 47514
Columns: 30

Driving style distribution:
DRIVING_STYLE
IDLE          20535
NORMAL        14505
ECO            8061
AGGRESSIVE     4413
Name: count, dtype: int64

Vehicles: 14


In [12]:
# Fill last remaining nulls with Unknown
df['MARK']         = df['MARK'].fillna('Unknown')
df['MODEL']        = df['MODEL'].fillna('Unknown')
df['CAR_YEAR']     = df['CAR_YEAR'].fillna(0)
df['ENGINE_POWER'] = df['ENGINE_POWER'].fillna(0)
df['AUTOMATIC']    = df['AUTOMATIC'].fillna('Unknown')

# Final save
df.to_csv(
    r'C:\Users\AmalDev\Downloads\obd-driver-behavior\obd_cleaned.csv',
    index=False
)

print("Total nulls:", df.isnull().sum().sum())
print("Shape:", df.shape)
print("\n✅ CLEANING PHASE 100% COMPLETE")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"Started with : 60,439 rows | 33 columns | 18 mixed-type columns")
print(f"Finished with: {len(df):,} rows | {len(df.columns)} columns | 0 nulls")
print(f"Rows removed : {60439 - len(df):,} ghost rows (no vehicle ID)")
print(f"Columns fixed: timestamps, % strings, European decimals, DTC codes")
print(f"New features : DRIVING_STYLE, FAULT_COUNT, IS_IDLE, RPM_PER_SPEED")

Total nulls: 0
Shape: (47514, 30)

✅ CLEANING PHASE 100% COMPLETE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Started with : 60,439 rows | 33 columns | 18 mixed-type columns
Finished with: 47,514 rows | 30 columns | 0 nulls
Rows removed : 12,925 ghost rows (no vehicle ID)
Columns fixed: timestamps, % strings, European decimals, DTC codes
New features : DRIVING_STYLE, FAULT_COUNT, IS_IDLE, RPM_PER_SPEED
